# Lab 3 — Does AI Beat the Decline Curve?

**Module 3.3 — Reservoir Characterization and Forecasting**

The deck makes a specific claim: *"AI often outperforms traditional decline curve methods."*

You are going to test it. Not discuss it — test it, on 768 real wells.

**Data:** North Dakota Bakken. Every producing well reported to the state, 2021–2024, filtered to
768 wells whose first production falls inside the window so you can see their whole early life.
Public records published by the North Dakota Department of Mineral Resources.

**Run each cell with Shift+Enter.** You are deciding which method you would stake a reserves
estimate on.


> **Kernel:** use **conda_python3**. The cell below installs anything missing.


In [ ]:
import subprocess, sys

for module, package in [("pandas", "pandas"), ("matplotlib", "matplotlib"),
                        ("sklearn", "scikit-learn"), ("scipy", "scipy")]:
    try:
        __import__(module)
    except ImportError:
        print(f"installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

print("Ready.")


## Step 1 — Real production histories


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("data/nd_bakken_production.csv", parse_dates=["REPORT_DATE"])

print(f"{len(df):,} well-months from {df.API.nunique()} wells")
print(f"{df.REPORT_DATE.min():%b %Y} to {df.REPORT_DATE.max():%b %Y}")
df.head()


### Why we forecast rate per producing day, not monthly volume

A well that produced 20,000 bbl in 31 days and one that produced 20,000 bbl in 12 days are not the
same well. Monthly totals mix **reservoir decline** with **downtime**. Dividing by `DAYS_PRODUCED`
gives bbl/day and strips the downtime out.

`OIL_BOPD` in this file is already that division. `MONTHS_ON_PROD` counts from each well's first
production, so wells that came online in different months can be compared directly.


In [ ]:
example = df[df.API == df.API.iloc[0]].sort_values("MONTHS_ON_PROD")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(example.MONTHS_ON_PROD, example.OIL_BBL, marker="o", ms=3)
axes[0].set_title("Monthly volume (mixes in downtime)")
axes[0].set_ylabel("bbl / month")
axes[1].plot(example.MONTHS_ON_PROD, example.OIL_BOPD, marker="o", ms=3, color="darkgreen")
axes[1].set_title("Rate per producing day (what we forecast)")
axes[1].set_ylabel("bbl / day")
for ax in axes:
    ax.set_xlabel("Months on production")
    ax.grid(alpha=0.3)
fig.suptitle(f"{example.WELL_NAME.iloc[0]} — {example.FIELD.iloc[0]} field", y=1.02)
plt.tight_layout()
plt.show()


That shape — a sharp peak then steep decline — is the signature of an unconventional well. Most of
the oil arrives early, which is exactly why forecasting the tail accurately matters so much to the
economics.


## Step 2 — The contest


Both methods get **the same information**: each well's first **18 months**. Both must predict
months **19 to 30**, which neither has seen.

**Method A — Arps decline curve.** Published by J.J. Arps in 1945. Three parameters, fitted per
well:

$$q(t) = \frac{q_i}{(1 + b \, d_i \, t)^{1/b}}$$

**Method B — Random forest**, fitted per well on the same 18 months.

We score with **MAPE** — mean absolute percentage error. Lower is better.


In [ ]:
def arps(t, qi, di, b):
    """Arps hyperbolic decline: initial rate qi, decline di, curvature b."""
    return qi / np.power(1 + b * di * t, 1 / np.maximum(b, 1e-6))


def mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / actual)) * 100


FIT_MONTHS = 18      # what each method is allowed to see
FORECAST_MONTHS = 12 # what it must predict

print(f"Fit on months 0-{FIT_MONTHS - 1}, forecast months "
      f"{FIT_MONTHS}-{FIT_MONTHS + FORECAST_MONTHS - 1}")


In [ ]:
results = []

for api, well in df.groupby("API"):
    well = well.sort_values("MONTHS_ON_PROD")
    well = well[well.MONTHS_ON_PROD < FIT_MONTHS + FORECAST_MONTHS]

    history = well[well.MONTHS_ON_PROD < FIT_MONTHS]
    future = well[well.MONTHS_ON_PROD >= FIT_MONTHS]
    if len(history) < 12 or len(future) < 6:
        continue

    # Decline is measured from peak rate, not from first production.
    peak_month = history.loc[history.OIL_BOPD.idxmax(), "MONTHS_ON_PROD"]
    fit = history[history.MONTHS_ON_PROD >= peak_month]
    if len(fit) < 8:
        continue

    try:
        params, _ = curve_fit(
            arps, fit.MONTHS_ON_PROD - peak_month, fit.OIL_BOPD,
            p0=[fit.OIL_BOPD.iloc[0], 0.15, 0.8],
            bounds=([0, 1e-4, 0.01], [1e6, 2.0, 2.0]), maxfev=8000)
    except Exception:
        continue

    dca_forecast = np.clip(arps(future.MONTHS_ON_PROD - peak_month, *params), 0.1, None)

    forest = RandomForestRegressor(n_estimators=60, random_state=0)
    forest.fit(fit[["MONTHS_ON_PROD"]], fit.OIL_BOPD)
    ml_forecast = np.clip(forest.predict(future[["MONTHS_ON_PROD"]]), 0.1, None)

    results.append({
        "API": api,
        "decline_curve": mape(future.OIL_BOPD.values, dca_forecast),
        "machine_learning": mape(future.OIL_BOPD.values, ml_forecast),
    })

scores = pd.DataFrame(results)
print(f"Forecast {len(scores)} wells.")


> ### ⏸ Predict before you look (2 min)
>
> The deck says AI often outperforms decline curves. **Which do you think won?**
>
> Say it out loud in your group before running the next cell.


In [ ]:
print(f"Arps decline curve (1945)   median error  {scores.decline_curve.median():5.1f}%")
print(f"Random forest               median error  {scores.machine_learning.median():5.1f}%")
print()
print(f"Decline curve was better on {(scores.decline_curve < scores.machine_learning).mean() * 100:.0f}% of wells")


> ### ⏸ Discuss (4 min)
>
> A three-parameter equation from 1945 beat the machine-learning model on most wells.
>
> - The deck's claim was not made up — AI *does* beat decline curves in some settings. **What is
>   different about this one?**
> - What is the random forest actually doing when asked about month 25, having only seen months
>   0–17?


## Step 3 — Look at what the model does


In [ ]:
api = df.API.iloc[0]
well = df[df.API == api].sort_values("MONTHS_ON_PROD")
well = well[well.MONTHS_ON_PROD < FIT_MONTHS + FORECAST_MONTHS]
history = well[well.MONTHS_ON_PROD < FIT_MONTHS]
future = well[well.MONTHS_ON_PROD >= FIT_MONTHS]

peak_month = history.loc[history.OIL_BOPD.idxmax(), "MONTHS_ON_PROD"]
fit = history[history.MONTHS_ON_PROD >= peak_month]

params, _ = curve_fit(arps, fit.MONTHS_ON_PROD - peak_month, fit.OIL_BOPD,
                      p0=[fit.OIL_BOPD.iloc[0], 0.15, 0.8],
                      bounds=([0, 1e-4, 0.01], [1e6, 2.0, 2.0]), maxfev=8000)

forest = RandomForestRegressor(n_estimators=60, random_state=0)
forest.fit(fit[["MONTHS_ON_PROD"]], fit.OIL_BOPD)

months = np.arange(peak_month, FIT_MONTHS + FORECAST_MONTHS)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(history.MONTHS_ON_PROD, history.OIL_BOPD, "o-", color="black", ms=4, label="History (given to both)")
ax.plot(future.MONTHS_ON_PROD, future.OIL_BOPD, "o-", color="gray", ms=4, label="Actual (hidden)")
ax.plot(months, arps(months - peak_month, *params), lw=2, color="seagreen", label="Arps decline curve")
ax.plot(months, forest.predict(pd.DataFrame({"MONTHS_ON_PROD": months})), lw=2, color="crimson", label="Random forest")
ax.axvline(FIT_MONTHS, ls="--", color="steelblue", lw=1, label="Forecast starts")
ax.set_xlabel("Months on production")
ax.set_ylabel("Oil rate (bbl/day)")
ax.set_title(f"{well.WELL_NAME.iloc[0]} — forecasting beyond what either method has seen")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


> ### ⏸ Discuss (4 min) — this is the point of the lab
>
> Look at the red line after the dashed marker.
>
> - A tree model predicts by averaging training examples that look like the input. Asked about
>   month 25 when it has only seen months 0–17, **what is the closest thing it knows?**
> - A tree-based model can never output a value outside the range it was trained on. **What does
>   that mean for forecasting anything that keeps declining?**
> - The green curve keeps falling because the equation encodes an assumption about how reservoirs
>   behave. **Where did that assumption come from, and what happens if it is wrong?**


## Step 4 — Your decision


Agree as a group. 3 minutes.

1. You must book reserves on a well with 18 months of history. **Which method, and why?**
2. The deck says AI often outperforms decline curves. **Is that claim wrong, or just incomplete?**
3. Name one forecasting job in this dataset where you would expect ML to **win**.
4. What would you ask a vendor who showed you an AI production forecast?

Be ready to answer in 60 seconds.


---

### Optional — where machine learning does win

Single-well extrapolation is the worst possible use of a tree model. But there is a question decline
curves cannot answer at all:

**Before a well is drilled, it has no history to fit.** A decline curve has nothing to work with. A
model trained across many wells can predict the new well's peak rate from its operator, field and
pool — because it has seen hundreds of analogues.


In [ ]:
from sklearn.model_selection import train_test_split

peaks = (df[df.MONTHS_ON_PROD < 6]
         .groupby(["API", "OPERATOR", "FIELD", "POOL", "COUNTY"], as_index=False)
         .OIL_BOPD.max().rename(columns={"OIL_BOPD": "PEAK_BOPD"}))

X = pd.get_dummies(peaks[["OPERATOR", "FIELD", "POOL", "COUNTY"]], drop_first=True)
y = peaks.PEAK_BOPD

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train, y_train)
predicted = model.predict(X_test)

field_average = np.full(len(y_test), y_train.mean())

print(f"Predicting peak rate for a well with NO production history:")
print(f"  Guess the average        median error {np.median(np.abs(y_test - field_average) / y_test) * 100:5.1f}%")
print(f"  Model (operator/field)   median error {np.median(np.abs(y_test - predicted) / y_test) * 100:5.1f}%")


**That** is the shape of problem machine learning is built for: many examples, interpolation rather
than extrapolation, and no physical equation available.

The honest version of the deck's claim is narrower and more useful:

> Machine learning outperforms decline curves when predicting **across wells**, where it can learn
> from analogues. Decline curves outperform machine learning when extrapolating **a single well
> forward in time**, because they encode physics and ML models cannot extrapolate past their
> training range.

Both tools are right for different questions. Knowing which question you are asking is the
reservoir engineer's job, not the model's.
